# llama.cpp on Kaggle (GPU)
1. Enable **GPU** accelerator (T4) + **Internet** in notebook settings.
2. Replace `YOUR_GITHUB_URL` and `MODEL_FILE` below, then run cells top to bottom.
3. Sessions are ephemeral — re-run this notebook each time. Keep 1–2 models per session (disk is ~20–30GB).

In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total --format=csv
git clone https://github.com/Rik-rut/Llama-CPP-Script-setup.git llama-setup
cd llama-setup
chmod +x setup.sh start-server.sh
./setup.sh

Download **one** model into `models/`. Worked example — Qwen3.6-27B-A3B-Coder at `Q4_K_M` (~16.7GB, swap `REPO_ID`/`FILE` for any other GGUF):

In [ ]:
REPO_ID = "ManniX-ITA/Qwen3.6-27B-A3B-Coder-MTP-GGUF"
FILE = "Qwen3.6-27B-A3B-Coder-Q4_K_M.gguf"

from huggingface_hub import snapshot_download
snapshot_download(repo_id=REPO_ID, local_dir="llama-setup/models",
                  allow_patterns=[FILE])
print("saved:", FILE)

Swap in the T4-tuned config, register this model (`filename|ctx|ngl|kv`), then launch in the background and wait for health:

⚠️ The Q4_K_M file is ~16.7GB — it does **not** fully fit a T4 16GB, so the line below uses partial offload (`ngl 45`, 32K ctx). It will run, but part of it executes on CPU. For full-GPU speed on a T4, pick a smaller tier from the same repo instead (e.g. `Q3_K_M` ~12.7GB or `IQ2_XS` ~8.2GB with `ngl 99`).

Author's sampler recipe for this model: `temp 0.6, top-k 20, top-p 0.95`, and bound its long thinking with per-request `thinking_budget_tokens: 8192` (server flags `--reasoning-budget 8192` also work if added to the launch command).

In [ ]:
%%bash
cd llama-setup
cp config/models.conf.kaggle-t4 config/models.conf
MODEL_FILE="Qwen3.6-27B-A3B-Coder-Q4_K_M.gguf"  # <-- must match FILE above
grep -q "$MODEL_FILE" config/models.conf || echo "$MODEL_FILE|32768|45|q8_0" >> config/models.conf
nohup ./start-server.sh "$MODEL_FILE" 18123 127.0.0.1 > logs/kaggle-server.log 2>&1 &
for i in $(seq 1 30); do curl -sf http://127.0.0.1:18123/health && break; sleep 10; done

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:18123/v1", api_key="sk-llama")
r = client.chat.completions.create(
    model="Local Model",
    messages=[{"role": "user", "content": "Reply with exactly: kaggle works"}],
    max_tokens=20,
)
print(r.choices[0].message.content)

### Optional: public URL via Cloudflare quick tunnel (outbound-only, works on Kaggle)
**Set `LLAMA_API_KEY` in `.env` first** and restart the server, or anyone with the URL can use your GPU.
Run the two cells below, then open the `*.trycloudflare.com` URL from the output.

Install cloudflared:

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Start the tunnel (prints the public URL — keep this cell running):

In [ ]:
import subprocess
import time

cloudflared = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url", "http://127.0.0.1:18123",
        "--http-host-header", "localhost:18123"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(8)

for _ in range(30):
    line = cloudflared.stdout.readline()
    if line:
        print(line, end="")

### Stop the server
`pkill -f llama-server`